# Olist E-Commerce &mdash; Dashboard Data Preprocessing

**Turns the 9 cleaned Olist tables into 9 dashboard-ready tables.**

This step runs **after** the lossless &ldquo;Clean data&rdquo; step. It reads the **cleaned**
tables, applies the preprocessing demand (steps **A&ndash;G**) to the 4 transaction
tables, and passes the 5 dimension tables through unchanged, so the dashboard has all 9.

**Input:** `cleaned/` (9 cleaned CSVs) &nbsp;&middot;&nbsp; **Output:** `processed/` (9 dashboard-ready CSVs)

Data lineage:

    raw (9)  ->  cleaned (9)  ->  processed (9)
                  Clean data step        Dashboard data step (this notebook)


## The preprocessing demand (steps A&ndash;G)

| Step | Table | Action |
|---|---|---|
| **A** | orders | Remove orders with timestamp inconsistencies (`delivered_carrier < approved` or `delivered_customer < delivered_carrier`). |
| **B** | reviews | Collapse to one row per order, keeping the **lowest** review score. |
| **C** | payments | Drop the invalid rows where `payment_value == 0` and `payment_type == not_defined`. |
| **D** | orders | Remove orders that have **no items**. |
| **E** | orders | Add time-series / delivery derived columns. |
| **F** | order_items | Add `item_revenue = price + freight_value`. |
| **G** | orders | Remove **all incomplete months** (keep 2017-01 &hellip; 2018-08). |

The 5 dimension tables (customers, sellers, products, geolocation, translation) are
**passed through unchanged** from `cleaned/`.


In [1]:
from pathlib import Path
import shutil

import pandas as pd


# Locate this notebook's own folder: prefer VS Code's notebook path, else the
# Jupyter cwd (= notebook folder), else walk up / one level down for the folder
# that has cleaned/ but no raw/.
def notebook_dir() -> Path:
    p = globals().get("__vsc_ipynb_file__")
    if p:
        return Path(p).resolve().parent

    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "cleaned" / "olist_orders_dataset.csv").exists() and not (d / "raw").exists():
            return d
    for d in start.iterdir():
        if d.is_dir() and (d / "cleaned" / "olist_orders_dataset.csv").exists() and not (d / "raw").exists():
            return d
    return start


BASE = notebook_dir()
CLEANED = BASE / "cleaned"
PROC = BASE / "processed"
PROC.mkdir(exist_ok=True)

print("Notebook folder :", BASE)
print("Cleaned input   :", CLEANED)
print("Processed output:", PROC)

assert CLEANED.exists(), f"cleaned/ not found next to the notebook. Expected: {CLEANED}"

pd.set_option("display.max_columns", None)


Notebook folder : E:\NUS最后一学期\IT5006\小组作业数据集\Dashboard data
Cleaned input   : E:\NUS最后一学期\IT5006\小组作业数据集\Dashboard data\cleaned
Processed output: E:\NUS最后一学期\IT5006\小组作业数据集\Dashboard data\processed


In [2]:
# Load the 4 transaction tables from cleaned/.
orders = pd.read_csv(
    CLEANED / "olist_orders_dataset.csv",
    dtype={"order_id": "string", "customer_id": "string", "order_status": "string"},
    parse_dates=["order_purchase_timestamp", "order_approved_at",
                 "order_delivered_carrier_date", "order_delivered_customer_date",
                 "order_estimated_delivery_date"],
)
items = pd.read_csv(
    CLEANED / "olist_order_items_dataset.csv",
    dtype={"order_id": "string", "order_item_id": "int64",
           "product_id": "string", "seller_id": "string",
           "price": "float64", "freight_value": "float64"},
    parse_dates=["shipping_limit_date"],
)
payments = pd.read_csv(
    CLEANED / "olist_order_payments_dataset.csv",
    dtype={"order_id": "string", "payment_sequential": "int64",
           "payment_type": "string", "payment_installments": "int64",
           "payment_value": "float64"},
)
reviews = pd.read_csv(
    CLEANED / "olist_order_reviews_dataset.csv",
    dtype={"review_id": "string", "order_id": "string",
           "review_score": "int64", "review_comment_title": "string",
           "review_comment_message": "string"},
    parse_dates=["review_creation_date", "review_answer_timestamp"],
)

print("Loaded rows ->  orders:", len(orders), "| items:", len(items),
      "| payments:", len(payments), "| reviews:", len(reviews))


Loaded rows ->  orders: 99441 | items: 112650 | payments: 103886 | reviews: 99224


## Orders &mdash; steps A, D, G (row removal)

Applied in order: **A** timestamp fix &rarr; **D** no-item removal &rarr; **G** incomplete months.


In [3]:
def report(step, before, after):
    print(f"{step:42s} {before:>9,} -> {after:>9,}  (removed {before - after:>7,})")


# A: drop orders whose carrier/customer delivery is earlier than approval.
n = len(orders)
bad = (orders["order_delivered_carrier_date"] < orders["order_approved_at"]) | (
    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
)
orders = orders[~bad]
report("A remove timestamp-inconsistent", n, len(orders))

# D: drop orders with no items.
n = len(orders)
orders = orders[orders["order_id"].isin(items["order_id"])]
report("D remove orders with no items", n, len(orders))

# G: keep only complete months 2017-01 .. 2018-08.
n = len(orders)
orders = orders[(orders["order_purchase_timestamp"] >= "2017-01-01") &
                (orders["order_purchase_timestamp"] < "2018-09-01")]
report("G remove incomplete months", n, len(orders))


A remove timestamp-inconsistent               99,441 ->    98,059  (removed   1,382)
D remove orders with no items                 98,059 ->    97,284  (removed     775)
G remove incomplete months                    97,284 ->    96,975  (removed     309)


## Propagate the order filter to items / payments / reviews

After A&middot;D&middot;G, drop every row in the other fact tables whose `order_id` no
longer exists in `orders`, so the tables stay referentially consistent.


In [4]:
keep = set(orders["order_id"])

items = items[items["order_id"].isin(keep)]
payments = payments[payments["order_id"].isin(keep)]
reviews = reviews[reviews["order_id"].isin(keep)]

print("after order filter -> items:", len(items), "| payments:", len(payments),
      "| reviews:", len(reviews))


after order filter -> items: 110683 | payments: 101297 | reviews: 96783


## Reviews &mdash; step B (one row per order, lowest score)

547 orders have more than one review. Keep the **lowest** review score per order;
for the comment, keep the comment of the lowest-scored review (ties broken by the
earliest `review_creation_date`).


In [5]:
reviews = (
    reviews
    .sort_values(["order_id", "review_score", "review_creation_date"])
    .groupby("order_id", as_index=False)
    .first()
)
print("reviews after B (one per order, lowest score):", len(reviews))


reviews after B (one per order, lowest score):

 96240


## Payments &mdash; step C (drop invalid zero payments)

3 rows have `payment_value == 0` and `payment_type == not_defined`; their orders are
`canceled` with no items, so they are invalid. (These rows are also removed by the
order filter above &mdash; this step makes the rule explicit.)


In [6]:
n = len(payments)
payments = payments[~((payments["payment_value"] == 0) & (payments["payment_type"] == "not_defined"))]
print("C payments: removed", n - len(payments), "invalid zero rows ->", len(payments), "remain")


C payments: removed 0 invalid zero rows -> 101297 remain


## Order items &mdash; step F (item revenue + ship-on-time)

`item_revenue = price + freight_value`. We also compute `is_on_time_shipped`
(carrier received the parcel before the item&rsquo;s `shipping_limit_date`) here,
because `shipping_limit_date` lives on the items table, not the orders table.


In [7]:
items = items.merge(
    orders[["order_id", "order_delivered_carrier_date"]],
    on="order_id", how="left",
)
items["item_revenue"] = items["price"] + items["freight_value"]
items["is_on_time_shipped"] = items["order_delivered_carrier_date"] <= items["shipping_limit_date"]
items = items.drop(columns=["order_delivered_carrier_date"])
print("order_items after F:", len(items), "rows | new cols: item_revenue, is_on_time_shipped")


order_items after F: 110683 rows | new cols: item_revenue, is_on_time_shipped


## Orders &mdash; step E (time-series / delivery columns)

Add `order_date`, `year_month`, `year`, `month`, `day_of_week`, `year_week`,
`delivery_days`, and `is_on_time`. (`is_on_time_shipped` is added to `order_items`
above, since it needs the item-level `shipping_limit_date`.)

> `delivery_days` / `is_on_time` are only meaningful for orders that have actually
> been delivered; for undelivered orders they are empty / `False`.


In [8]:
iso_cal = orders["order_purchase_timestamp"].dt.isocalendar()

orders["order_date"] = orders["order_purchase_timestamp"].dt.date
orders["year_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders["year"] = orders["order_purchase_timestamp"].dt.year
orders["month"] = orders["order_purchase_timestamp"].dt.month
orders["day_of_week"] = orders["order_purchase_timestamp"].dt.day_name()
orders["year_week"] = iso_cal["year"].astype(str) + "-" + iso_cal["week"].astype(str).str.zfill(2)
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["is_on_time"] = orders["order_delivered_customer_date"] <= orders["order_estimated_delivery_date"]

print("orders after E:", len(orders), "rows |", len(orders.columns), "columns")
print(orders[["order_id", "year_month", "day_of_week", "year_week",
              "delivery_days", "is_on_time"]].head())


orders after E: 96975 rows | 16 columns
                           order_id year_month day_of_week year_week  \
0  e481f51cbdc54678b7cc49136f2d6af7    2017-10      Monday   2017-40   
1  53cdb2fc8bc7dce0b6741e2150273451    2018-07     Tuesday   2018-30   
2  47770eb9100c2d0c44946d9cf07ec65d    2018-08   Wednesday   2018-32   
3  949d5b44dbf5de918fe9c16f97b45f8a    2017-11    Saturday   2017-46   
4  ad21c59c0840e6cb83a9ceb5573f8159    2018-02     Tuesday   2018-07   

   delivery_days  is_on_time  
0            8.0        True  
1           13.0        True  
2            9.0        True  
3           13.0        True  
4            2.0        True  


## Write the 4 transformed transaction tables


In [9]:
out = {
    "orders": (orders, "olist_orders_dataset.csv"),
    "order_items": (items, "olist_order_items_dataset.csv"),
    "payments": (payments, "olist_order_payments_dataset.csv"),
    "reviews": (reviews, "olist_order_reviews_dataset.csv"),
}

summary = []
for name, (df, fn) in out.items():
    df.to_csv(PROC / fn, index=False, encoding="utf-8")
    summary.append((name, fn, len(df)))
    print(f"[OK] {fn:36s} {len(df):>8,} rows -> processed/")

summary_df = pd.DataFrame(summary, columns=["table", "file", "rows"])
summary_df


[OK] olist_orders_dataset.csv               96,975 rows -> processed/


[OK] olist_order_items_dataset.csv         110,683 rows -> processed/
[OK] olist_order_payments_dataset.csv      101,297 rows -> processed/


[OK] olist_order_reviews_dataset.csv        96,240 rows -> processed/


,table,file,rows
0,orders,olist_orders_dataset.csv,96975
1,order_items,olist_order_items_dataset.csv,110683
2,payments,olist_order_payments_dataset.csv,101297
3,reviews,olist_order_reviews_dataset.csv,96240


## Dimension tables &mdash; passed through unchanged

The 5 dimension tables (customers, sellers, products, geolocation, translation) are
already clean from the &ldquo;Clean data&rdquo; step (typed, zip leading zeros preserved,
geolocation deduplicated). They are copied verbatim so the dashboard has all 9 tables.


In [10]:
DIMENSIONS = [
    "olist_customers_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_products_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv",
]

for fn in DIMENSIONS:
    shutil.copy(CLEANED / fn, PROC / fn)
    print(f"[OK] {fn:36s} passed through (unchanged) -> processed/")


[OK] olist_customers_dataset.csv          passed through (unchanged) -> processed/
[OK] olist_sellers_dataset.csv            passed through (unchanged) -> processed/
[OK] olist_products_dataset.csv           passed through (unchanged) -> processed/
[OK] olist_geolocation_dataset.csv        passed through (unchanged) -> processed/
[OK] product_category_name_translation.csv passed through (unchanged) -> processed/


## Verification (referential integrity + demand rules + dimensions)


In [11]:
oids = set(orders["order_id"])
iids = set(items["order_id"])

checks = {
    "every order_item.order_id is in orders": items["order_id"].isin(oids).all(),
    "every payment.order_id is in orders": payments["order_id"].isin(oids).all(),
    "every review.order_id is in orders": reviews["order_id"].isin(oids).all(),
    "every order has >=1 item": orders["order_id"].isin(iids).all(),
    "no timestamp-inconsistent orders": not (
        (orders["order_delivered_carrier_date"] < orders["order_approved_at"]) |
        (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"])
    ).any(),
    "months all within 2017-01..2018-08": (
        (orders["order_purchase_timestamp"] >= "2017-01-01") &
        (orders["order_purchase_timestamp"] < "2018-09-01")
    ).all(),
    "no invalid zero payments": not (
        (payments["payment_value"] == 0) & (payments["payment_type"] == "not_defined")
    ).any(),
    "reviews one row per order": reviews["order_id"].is_unique,
}

# dimension pass-through: processed == cleaned (same row count, all 5 present)
for fn in DIMENSIONS:
    cn = pd.read_csv(CLEANED / fn, usecols=[0]).shape[0]
    pn = pd.read_csv(PROC / fn, usecols=[0]).shape[0]
    checks[f"dimension {fn} lossless"] = cn == pn

all_ok = all(checks.values())
for k, v in checks.items():
    print(f"  [{'OK ' if v else 'FAIL'}] {k}")
print()
print("RESULT:", "ALL CHECKS PASSED" if all_ok else "CHECK FAILED")


  [OK ] every order_item.order_id is in orders
  [OK ] every payment.order_id is in orders
  [OK ] every review.order_id is in orders
  [OK ] every order has >=1 item
  [OK ] no timestamp-inconsistent orders
  [OK ] months all within 2017-01..2018-08
  [OK ] no invalid zero payments
  [OK ] reviews one row per order
  [OK ] dimension olist_customers_dataset.csv lossless
  [OK ] dimension olist_sellers_dataset.csv lossless
  [OK ] dimension olist_products_dataset.csv lossless
  [OK ] dimension olist_geolocation_dataset.csv lossless
  [OK ] dimension product_category_name_translation.csv lossless

RESULT: ALL CHECKS PASSED
